# Meridian Health Dataset — Data Cleaning

**Digitera Program | Data Analysis Track**

This notebook cleans the Meridian Health dataset using Python and pandas.

### Tasks covered
1. Load the CSV files
2. Combine the three sales files
3. Format `sale_date`
4. Clean text prices
5. Normalize doctor names and dosage
6. Split `pack_info`
7. Extract promo codes
8. Join product cost and calculate financial metrics
9. Extract customer loyalty information
10. Clean customer names, phones, and duplicates


## 0. Setup

Put this notebook in the same folder as the Meridian Health CSV files.

In [ ]:
import pandas as pd
import re
from pathlib import Path

DATA_FOLDER = Path('/content/Data')

print('Setup complete.')

Setup complete.


## 1. Helper Functions

In [1]:
def find_col(df, names):
    """Find a column using several possible names."""
    for name in names:
        if name in df.columns:
            return name
    return None


def clean_money(value):
    """Convert values such as '$123.45' to numbers."""
    if pd.isna(value):
        return None
    value = str(value).replace('$', '').replace(',', '').strip()
    return pd.to_numeric(value, errors='coerce')


def normalize_text(value):
    if pd.isna(value):
        return value
    return str(value).strip().title()


def normalize_dosage(value):
    if pd.isna(value):
        return value
    value = str(value).strip().lower()
    replacements = {
        'mg.': 'mg', 'milligram': 'mg', 'milligrams': 'mg',
        'ml.': 'ml', 'milliliter': 'ml', 'milliliters': 'ml',
        'mcg.': 'mcg', 'microgram': 'mcg', 'micrograms': 'mcg'
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    return value


def extract_promo(value):
    if pd.isna(value):
        return None
    match = re.search(
        r'\b(?:PROMO|PROMO[-_ ]?CODE|CODE)[-_ ]?([A-Z0-9]+)\b',
        str(value).upper()
    )
    return match.group(1) if match else None


def split_pack_info(value):
    if pd.isna(value):
        return pd.Series([None, None, None])

    text = str(value).strip()

    strength = re.search(
        r'(\d+(?:\.\d+)?)\s*(mg|mcg|g|kg|ml|l)\b', text, re.I
    )
    strength_value = strength.group(0) if strength else None

    pack = re.search(
        r'(\d+)\s*(tablets?|capsules?|tabs?|caps?|bottles?|packs?|pieces?)',
        text, re.I
    )
    pack_value = pack.group(0) if pack else None

    forms = [
        'tablet', 'tablets', 'capsule', 'capsules', 'syrup',
        'cream', 'ointment', 'injection', 'solution', 'drops',
        'gel', 'spray'
    ]
    form_value = next((x for x in forms if x in text.lower()), None)

    return pd.Series([pack_value, strength_value, form_value])

print('Helper functions ready.')

Helper functions ready.


## 2. Load the Meridian Health CSV Files

In [ ]:
csv_files = list(DATA_FOLDER.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError(
        'No CSV files found. Put the Meridian Health CSV files in the same folder as this notebook.'
    )

tables = {}

for file in csv_files:
    try:
        tables[file.stem.lower()] = pd.read_csv(file)
        print(f'Loaded: {file.name} | Rows: {len(tables[file.stem.lower()])}')
    except Exception as e:
        print(f'Could not read {file.name}: {e}')

print(f'\nTotal CSV files loaded: {len(tables)}')

Loaded: sales_group3_storesC.csv | Rows: 3373
Loaded: customers_master.csv | Rows: 1555
Loaded: stores.csv | Rows: 12
Loaded: categories.csv | Rows: 9
Loaded: suppliers.csv | Rows: 5
Loaded: sales_group2_storesB.csv | Rows: 3325
Loaded: stock_condition_log.csv | Rows: 10000
Loaded: products_catalog.csv | Rows: 5000
Loaded: sales_group1_storesA.csv | Rows: 3252
Loaded: prescriptions_log.csv | Rows: 10000
Loaded: employees.csv | Rows: 120

Total CSV files loaded: 11


## 3. Combine the Three Sales Files

In [ ]:
sales_candidates = []

for name, df in tables.items():
    if any(word in name for word in ['sales', 'sale']):
        temp = df.copy()
        rename_map = {}

        qty_col = find_col(temp, ['qty', 'quantity', 'units', 'sold_qty'])
        if qty_col:
            rename_map[qty_col] = 'quantity'

        amount_col = find_col(
            temp, ['amount', 'unit_price', 'price', 'sale_price', 'unitprice']
        )
        if amount_col:
            rename_map[amount_col] = 'unit_price'

        date_col = find_col(
            temp, ['sale_date', 'date', 'sales_date', 'transaction_date']
        )
        if date_col:
            rename_map[date_col] = 'sale_date'

        temp = temp.rename(columns=rename_map)
        sales_candidates.append(temp)
        print(f'Added sales file: {name}')

if not sales_candidates:
    raise ValueError('No sales files were detected.')

sales = pd.concat(sales_candidates, ignore_index=True, sort=False)

print(f'\nCombined sales rows: {len(sales)}')
print('Columns:')
print(list(sales.columns))

Added sales file: sales_group3_storesc
Added sales file: sales_group2_storesb
Added sales file: sales_group1_storesa

Combined sales rows: 9950
Columns:
['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id', 'quantity', 'sale_date', 'unit_price', 'discount_pct', 'payment_method', 'notes']


## 4. Format `sale_date` as DD/MM/YYYY

In [ ]:
if 'sale_date' in sales.columns:
    sales['sale_date'] = pd.to_datetime(
        sales['sale_date'], errors='coerce'
    ).dt.strftime('%d/%m/%Y')

print(sales[['sale_date']].head() if 'sale_date' in sales.columns else 'sale_date column not found.')

    sale_date
0  06/10/2025
1  09/12/2025
2  18/08/2025
3  12/12/2024
4  02/08/2025


## 5. Standardize Prices

In [ ]:
for col in ['unit_price', 'amount', 'cost_price']:
    if col in sales.columns:
        sales[col] = sales[col].apply(clean_money)

print('Price cleaning complete.')

Price cleaning complete.


## 6. Normalize Doctor Names and Dosage

In [ ]:
doctor_col = find_col(
    sales, ['doctor_name', 'doctor', 'physician', 'prescribing_doctor']
)

if doctor_col:
    sales['doctor_name'] = sales[doctor_col].apply(normalize_text)

dosage_col = find_col(sales, ['dosage', 'dose', 'strength'])

if dosage_col:
    sales['dosage'] = sales[dosage_col].apply(normalize_dosage)

print('Doctor/dosage cleaning complete.')

Doctor/dosage cleaning complete.


## 7. Split `pack_info` into Pack Size, Strength, and Form

In [ ]:
pack_col = find_col(sales, ['pack_info', 'pack', 'package'])

if pack_col:
    sales[['Pack Size', 'Strength', 'Form']] = sales[pack_col].apply(split_pack_info)

print('Pack information processed.')

Pack information processed.


## 8. Extract Promo Codes from Sales Notes

In [ ]:
notes_col = find_col(
    sales, ['sales_notes', 'sale_notes', 'notes', 'sales_note']
)

if notes_col:
    sales['promo_code'] = sales[notes_col].apply(extract_promo)

print('Promo code extraction complete.')

Promo code extraction complete.


## 9. Join Product Cost and Calculate Financial Metrics

In [ ]:
product_df = None

for name, df in tables.items():
    if any(word in name for word in ['product', 'catalog']):
        product_df = df.copy()
        print(f'Product catalog found: {name}')
        break

if product_df is not None:
    product_id_sales = find_col(
        sales, ['product_id', 'product_code', 'sku', 'product']
    )
    product_id_catalog = find_col(
        product_df, ['product_id', 'product_code', 'sku', 'product']
    )
    cost_col = find_col(product_df, ['cost_price', 'cost', 'unit_cost'])

    if product_id_sales and product_id_catalog and cost_col:
        product_df[cost_col] = product_df[cost_col].apply(clean_money)

        catalog_small = product_df[[product_id_catalog, cost_col]].drop_duplicates()
        catalog_small = catalog_small.rename(columns={
            product_id_catalog: product_id_sales,
            cost_col: 'cost_price'
        })

        # Avoid creating duplicate cost_price columns if one already exists.
        if 'cost_price' in sales.columns:
            sales = sales.drop(columns=['cost_price'])

        sales = sales.merge(catalog_small, on=product_id_sales, how='left')

if 'quantity' in sales.columns:
    sales['quantity'] = pd.to_numeric(sales['quantity'], errors='coerce')

if 'unit_price' in sales.columns and 'quantity' in sales.columns:
    sales['sales_amount'] = sales['quantity'] * sales['unit_price']

if 'cost_price' in sales.columns and 'quantity' in sales.columns:
    sales['cost_amount'] = sales['quantity'] * sales['cost_price']

if 'sales_amount' in sales.columns and 'cost_amount' in sales.columns:
    sales['profit'] = sales['sales_amount'] - sales['cost_amount']
    sales['margin_pct'] = (
        sales['profit'] / sales['sales_amount'] * 100
    ).replace([float('inf'), -float('inf')], None)

print('Financial calculations complete.')
sales.head()

Product catalog found: products_catalog
Financial calculations complete.


,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,promo_code,cost_price,sales_amount,cost_amount,profit,margin_pct
0,2,11,44,4308,1300,3,06/10/2025,56.69,29.6,Debit Card,NaN,None,36.82,170.07,110.46,59.61,35.050273
1,3,11,63,2359,586,5,09/12/2025,104.55,33.7,Credit Card,NaN,None,73.40,522.75,367.00,155.75,29.794357
2,4,10,33,44,1381,2,18/08/2025,38.39,25.6,Credit Card,NaN,None,16.10,76.78,32.20,44.58,58.061995
3,5,12,60,3991,146,5,12/12/2024,21.74,4.6,Debit Card,Promo code LOYALTY20 applied at checkout,CODE,9.23,108.70,46.15,62.55,57.543698
4,6,10,16,1274,158,5,02/08/2025,102.58,32.3,Cash,NaN,None,59.25,512.90,296.25,216.65,42.240203


## 10. Extract Customer Loyalty Information and Clean Customers

In [ ]:
customer_df = None

for name, df in tables.items():
    if 'customer' in name:
        customer_df = df.copy()
        print(f'Customer file found: {name}')
        break

if customer_df is not None:
    text_columns = customer_df.select_dtypes(include='object').columns

    def extract_loyalty(row):
        text = ' '.join(
            str(row[col]) for col in text_columns if pd.notna(row[col])
        ).lower()

        if 'gold' in text:
            return 'Gold'
        if 'silver' in text:
            return 'Silver'
        if 'bronze' in text:
            return 'Bronze'
        if 'loyal' in text:
            return 'Loyal'
        if 'member' in text:
            return 'Member'
        return None

    customer_df['loyalty_status'] = customer_df.apply(extract_loyalty, axis=1)

    name_col = find_col(
        customer_df, ['customer_name', 'name', 'full_name', 'customer']
    )
    phone_col = find_col(
        customer_df, ['phone', 'phone_number', 'mobile', 'mobile_number']
    )

    if name_col:
        customer_df['customer_name'] = (
            customer_df[name_col]
            .astype('string')
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
            .str.title()
        )
        customer_df['name_clean'] = customer_df['customer_name']

    if phone_col:
        customer_df['phone_clean'] = (
            customer_df[phone_col]
            .astype('string')
            .str.replace(r'\D', '', regex=True)
        )

    # Remove exact duplicate rows.
    customer_df = customer_df.drop_duplicates()

    # Remove duplicate people using phone when available.
    if 'phone_clean' in customer_df.columns:
        customer_df = customer_df.drop_duplicates(
            subset=['phone_clean'], keep='first'
        )
    elif 'customer_name' in customer_df.columns:
        customer_df = customer_df.drop_duplicates(
            subset=['customer_name'], keep='first'
        )

    print(f'Clean customer rows: {len(customer_df)}')
    customer_df.head()

Customer file found: customers_master
Clean customer rows: 1513


## 11. Quick Data Check

In [ ]:
print('Sales shape:', sales.shape)
print('\nSales columns:')
print(list(sales.columns))

print('\nMissing values in sales:')
print(sales.isna().sum().sort_values(ascending=False).head(15))

if customer_df is not None:
    print('\nCustomer shape:', customer_df.shape)
    print('\nMissing values in customers:')
    print(customer_df.isna().sum().sort_values(ascending=False).head(15))

Sales shape: (9950, 17)

Sales columns:
['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id', 'quantity', 'sale_date', 'unit_price', 'discount_pct', 'payment_method', 'notes', 'promo_code', 'cost_price', 'sales_amount', 'cost_amount', 'profit', 'margin_pct']

Missing values in sales:
promo_code        9095
notes             8294
sale_date         3252
sale_id              0
store_id             0
customer_id          0
quantity             0
product_id           0
employee_id          0
discount_pct         0
unit_price           0
payment_method       0
cost_price           0
sales_amount         0
cost_amount          0
dtype: int64

Customer shape: (1513, 11)

Missing values in customers:
loyalty_status    1070
notes              837
customer_id          0
email                0
full_name            0
city                 0
phone                0
signup_date          0
customer_name        0
name_clean           0
phone_clean          0
dtype: int64


## 12. Save the Cleaned Files

In [ ]:
sales.to_csv('cleaned_sales.csv', index=False)

if customer_df is not None:
    customer_df.to_csv('cleaned_customers.csv', index=False)

print('DONE!')
print('Created: cleaned_sales.csv')
if customer_df is not None:
    print('Created: cleaned_customers.csv')

DONE!
Created: cleaned_sales.csv
Created: cleaned_customers.csv
